# Recent Feature Additions

## Version 0.4.0

### AC OPF Performance: AC PF Warm-Start + Interior-Point NR

The AC OPF solver now uses AC PF (Newton-Raphson) as its warm-start, replacing the previous flat-start approach. This fixes convergence failures on meshed/cyclic networks (e.g., IEEE-13) and dramatically improves performance:

| Model | Before | After |
|-------|--------|-------|
| p5r (radial) | 13 iter, 0.37s | 1 iter, 0.03s |
| IEEE-13 (meshed) | **FAIL** (50 iter) | 1 iter, 0.3s |

When the AC PF solution satisfies voltage bounds, it is accepted directly. If bounds are violated, an interior-point NR with log-barrier terms refines the solution while maintaining strict feasibility.

### MCP Server Migrated to v2

The MCP server now uses `mcp>=2.0.0` with the new `add_request_handler()` API:
- Types from `mcp_types` package
- `ListToolsRequest` / `CallToolRequest` handler registration
- `mcp.stdio_server` for transport

### Violation Detection & Automated Fix Engine

New `gdm_flow.fix` module provides iterative remediation of voltage and loading violations:

- **`detect_violations(system, solver="ldf")`** — runs power flow and identifies all out-of-band voltages and overloaded branches
- **`fix_violations(system)`** — iteratively applies strategies until violations are resolved or no progress is made

Four built-in strategies (applied in priority order):

| Strategy | Fixes | Action |
|----------|-------|--------|
| `AdjustRegulatorTapStrategy` | Voltage | Adjusts regulator tap setpoints |
| `AddCapacitorStrategy` | Undervoltage | Increases existing cap or flags new cap needed |
| `ResizeConductorStrategy` | Loading + undervoltage | Reduces branch impedance |
| `ResizeTransformerStrategy` | Loading | Upsizes to next standard kVA rating |

CLI command: `gdm-flow fix MODEL [--solver ldf] [--max-iter 10]`

### MCP `scale_loads` Tool

New MCP tool to uniformly scale all load P/Q before running solvers — useful for stress-testing at higher demand levels.

### Code Quality & CI

- **Codecov** integration with coverage upload on every push to `main`
- **89% coverage enforcement** in CI and pre-commit (pre-push stage)
- `dashboard.py` excluded from coverage (visualization-only code)
- Ruff lint + format enforced via pre-commit hooks
- All optional dependencies (`mcp`, `plotting`, `opendss`) installed in CI

## Version 0.3.0

### Split-phase (center-tapped transformer) support

All solvers now correctly handle center-tapped transformers and split-phase (S1/S2) buses:

- **Y-bus**: Kron-reduced 3-node admittance model with proper polarity — S1 in-phase with primary, S2 anti-phase. Includes S1-S2 coupling terms essential for NR convergence.
- **AC OPF & AC PF**: S1/S2 angle initialization propagates the primary phase angle through the secondary network via BFS, so transformers on phases B or C get correct starting angles.
- **DC OPF**: Linearization correction for S2 offset (negates `v_nom` for S2 buses).
- **LinDistFlow**: Phase angle map includes S1/S2, with center-tap detection and voltage-drop model for split-phase transformers.

### Automatic parallel transformer aggregation

The CLI auto-detects and aggregates parallel single-phase transformers and regulators:

- Groups of 3 single-phase units → 3-phase equivalent
- Groups of 2 single-phase units → open-wye/open-delta equivalent
- Emits warnings when aggregation is applied

### `DistributionRegulator` stamping in Y-bus

The Y-bus builder now stamps `DistributionRegulator` components (previously only `DistributionTransformer` was stamped), enabling correct voltage regulation modeling in NR-based solvers.

### LinDistFlow geometry conversion

Added `convert_geometry_to_matrix: bool = True` parameter to `solve_lindistflow()`. When enabled, geometry-based branches are automatically converted to matrix impedance form before solving. The AC PF warm-start call uses `convert_geometry_to_matrix=False` to avoid mutating the system.

### AC OPF NR warm-start always enabled

The Newton-Raphson warm-start (LinDistFlow → NR) is now always used regardless of system size, providing significant speedup on medium-sized systems (e.g., 135x on IEEE-123).

### CLI `compare` command enhancements

The `gdm-flow compare` command now produces rich diagnostic output:

| Feature | Description |
|---------|-------------|
| Voltage rows | V_min, V_max, V_mean (pu) and count of buses below 0.95 pu |
| Per-phase loading table | Load P/Q per phase with source injection from each solver |
| Disagreement metric | Max absolute difference in source P across solvers |
| HTML export (`-o`) | Voltage-distance scatter per phase, V vs P jointplot, source power bar chart |

In [ ]:
# Example: run solver comparison from Python
# gdm-flow compare tests/data/ieee-123/gdm/ieee_123_node.json -o report.html

### Solver results summary

Tested across three reference models:

| Model | AC OPF | AC PF | DC OPF | LDF | Max Disagreement |
|-------|--------|-------|--------|-----|------------------|
| IEEE-13 | 3.60 MW | 3.60 MW | 3.47 MW | 3.47 MW | 130 kW |
| IEEE-123 | 3.59 MW | 3.59 MW | 3.49 MW | 3.49 MW | 104 kW |
| P4U (split-phase) | 2.12 MW | 2.12 MW | 2.08 MW | 2.08 MW | 40 kW |